# Notebook 1 — Hierarchical Clustering Analysis (HCA)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/indicium15/ml-workshop/blob/main/workshop-2/notebooks/01_HCA_Hierarchical_Clustering.ipynb)

**Best used for:** discovering possible natural groups when you do not already know the categories.

## What is HCA?

**Hierarchical Clustering Analysis (HCA)** groups data points into a tree of nested clusters without requiring you to specify the number of groups in advance. It works by:

1. Treating each data point as its own cluster.
2. Repeatedly merging the two closest clusters until only one remains.
3. Recording the merge distances to produce a **dendrogram** — a tree diagram.

## Reading a Dendrogram

- The **height** of each horizontal merge line = the distance between the two clusters being joined.
- To choose *k* clusters, draw a horizontal cut line: the number of vertical lines it crosses = number of clusters.
- A **large vertical gap** before a merge suggests the clusters below are genuinely distinct.

We have survey and performance data for 500 university students. **Without any predefined categories**, can we discover natural *student archetypes* — groups of students who share similar study habits, engagement, and performance patterns?

This unsupervised approach is particularly valuable when you do not yet know what groupings exist in your data.

---

## Running This Notebook

You can use this notebook in either:

- **Google Colab**: best if you do not have Python installed.
- **Local Jupyter**: best if you already have the workshop folder on your computer.

### In Google Colab

1. Open this notebook from GitHub using the **Open in Colab** button.
2. Run the first setup cell.
3. Wait until it says the libraries and workshop files are loaded.
4. In **Step 1**, either use the pre-loaded sample dataset or upload your own CSV file.
5. Continue running each cell from top to bottom.

If a widget does not appear immediately in Colab, wait for the setup cell to finish, then rerun the current cell.

In [1]:
# SETUP
import sys, os, subprocess, importlib.util
from pathlib import Path

# Works locally from workshop-2/notebooks, from workshop-2, or in Google Colab
# after opening the notebook from GitHub.
REPO_URL = 'https://github.com/indicium15/ml-workshop.git'

def _find_workshop_root():
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd() / 'workshop-2',
        Path.cwd().parent / 'workshop-2',
        Path('/content/ml-workshop/workshop-2'),
        Path('/content/workshop-2'),
    ]
    for candidate in candidates:
        if (candidate / 'utils' / 'data_loader.py').exists():
            return candidate.resolve()
    return None

IN_COLAB = 'google.colab' in sys.modules
WORKSHOP_ROOT = _find_workshop_root()

if WORKSHOP_ROOT is None and IN_COLAB:
    print('Workshop files not found in Colab. Cloning the workshop repository...')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, '/content/ml-workshop'], check=True)
    WORKSHOP_ROOT = _find_workshop_root()

if WORKSHOP_ROOT is None:
    raise FileNotFoundError(
        'Could not find workshop-2/utils. Locally, start Jupyter from the workshop-2 folder. '
        'In Colab, upload the workshop-2 folder or open the notebook from the GitHub repository.'
    )

if str(WORKSHOP_ROOT) not in sys.path:
    sys.path.insert(0, str(WORKSHOP_ROOT))

required_modules = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'sklearn': 'scikit-learn',
    'scipy': 'scipy',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn',
    'ipywidgets': 'ipywidgets',
}
missing = [pip_name for module_name, pip_name in required_modules.items()
           if importlib.util.find_spec(module_name) is None]
if missing:
    print('Installing missing packages:', ', '.join(missing))
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(WORKSHOP_ROOT / 'requirements.txt')], check=True)

import ipywidgets as widgets

print(f'Using workshop files from: {WORKSHOP_ROOT}')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from IPython.display import display
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
import matplotlib.pyplot as plt
%matplotlib inline

from scipy.cluster.hierarchy import linkage, fcluster

from utils.data_loader import DataLoaderWidget
from utils.preprocessor import PreprocessorWidget
from utils.plotting import (
    plot_dendrogram,
    plot_cluster_scatter,
    plot_cluster_profile_heatmap,
    plot_correlation_heatmap,
)

print('Libraries loaded ✓')


Using workshop files from: /Users/chaitanya/Documents/Coding/ml-workshop/workshop-2
Libraries loaded ✓


### Colab Notes

- The first setup cell may take a minute because it checks for required packages.
- Uploaded files in Colab are temporary. If the runtime disconnects, upload the file again.
- To keep your own completed version, use **File → Save a copy in Drive**.
- If widgets do not display, run the setup cell again, then rerun the current cell.

## Step 1 — Load Data

The sample dataset is already available when you run the setup cell.

To use your own data in Colab:

1. Click the upload control.
2. Choose a `.csv` file from your computer.
3. Wait for the preview table to appear.
4. Select the columns you want to use.
5. Click **Confirm Selection**.

For local Jupyter, you can either upload a CSV or type a path to a CSV file on your machine.

### Recommended Selection for Workshop
- lecture_attendance_rate
- tutorial_attendance_rate
- tutorial_participation_score
- office_hours_visits
- lms_logins_per_week
- assignment_completion_rate
- self_reported_stress_level
- avg_weekly_study_hours
- part_time_work_hours


### What the Load Data controls do

| Control | What it does | When to adjust it |
|---|---|---|
| **CSV path** | Points the notebook to a CSV file on disk. The default path loads the workshop sample dataset. | Change this when running locally with your own CSV file. |
| **Load CSV** | Reads the file from **CSV path** and displays a short preview plus column counts. | Click this after editing the path. |
| **Or upload** | Lets you upload a CSV directly in Colab/Jupyter instead of typing a path. | Use this when your CSV is on your computer rather than already in the notebook folder. |
| **Features** | Selects the input columns the later analysis/model will use. Hold Ctrl/Cmd to select multiple columns. | Exclude IDs, labels/targets, free-text notes, or columns you do not want the method to learn from. |
| **Select all** | Selects every column except common ID/target columns. | Useful as a starting point, then remove anything inappropriate. |
| **Numeric only** | Selects only integer/float columns. | Useful for a simpler first run or for methods where you want to avoid categorical encoding. |
| **Confirm Selection** | Saves the selected features for later cells. | Click this before preprocessing or running any analysis. |

> Keep a copy of your original dataset unchanged. These notebooks do not edit the source CSV.

> **Tip:** For clustering, do not select ID columns (`student_id`) or known outcome labels such as `performance_band`. Use numeric or ordinal features that describe the rows you want to group.


In [ ]:
# LOAD DATA
recommended_features = [
    'lecture_attendance_rate',
    'tutorial_attendance_rate',
    'tutorial_participation_score',
    'office_hours_visits',
    'lms_logins_per_week',
    'assignment_completion_rate',
    'avg_weekly_study_hours',
    'self_reported_stress_level',
    'extracurricular_hours_per_week',
    'part_time_work_hours',
]
loader = DataLoaderWidget(
    show_label_selector=False,
    default_feature_columns=recommended_features,
)
loader.display()

## Step 2 — Preprocessing

Scaling is important for distance-based methods. **StandardScaler** (zero mean, unit variance) is recommended.

After clicking **Apply Preprocessing**, the scaled data is ready for clustering.

### What the Preprocessing controls do

| Control | What it does | When to adjust it |
|---|---|---|
| **Scaling** | Rescales numeric features before distance calculations. `standard` makes values comparable by centering and scaling; `minmax` maps values into a fixed 0-1 range; `none` leaves values unchanged. | Use `standard` for most clustering runs. Try `minmax` when bounded 0-1 features are easier to interpret. Avoid `none` unless all features are already on comparable scales. |
| **Categorical enc** | Converts text/category columns into numbers. `label` assigns each category an integer; `onehot` creates one column per category; `drop` removes categorical columns. | Use label for ordinal categories where the order matters, such as low < medium < high or education levels. Avoid it for nominal categories like faculty, gender, or accommodation type because the numeric codes can imply a false ranking. Use onehot for unordered categories, and use drop only when the column is irrelevant, too noisy, or would create too many features. |
| **Random state** | Sets the seed used by reproducible preprocessing steps. | Keep the default when comparing methods; change it only when testing sensitivity to random choices. |
| **Apply Preprocessing** | Builds the processed feature matrix used by the clustering cells. | Click after changing selected columns or preprocessing settings. |



In [3]:
# HYPERPARAMETERS
preprocessor = PreprocessorWidget(
    source_loader=loader,
    y=None,
    clustering_mode=True,
)
preprocessor.display()


## Step 2b — Feature Scale Check

Before looking at the dendrogram, it is worth checking the **range of values** across your features.

HCA uses Euclidean distances (with Ward linkage) or other distance metrics. If one feature spans
0-2400 and another spans 0-5, the large-scale feature will dominate every pairwise distance
calculation - the algorithm effectively ignores the small-scale features.

The chart below highlights this problem using two deliberately paired columns in the sample dataset:

| Column | Range | Represents |
|---|---|---|
| `avg_weekly_study_hours` | 0-40 | Same behaviour... |
| `total_study_minutes_per_week` | 0-2400 | ...just measured in minutes (x60) |
| `lms_logins_per_week` | 0-20 | Same behaviour... |
| `cumulative_lms_sessions_per_semester` | 0-300 | ...accumulated over a semester (x13) |

> **Without scaling**, HCA cluster boundaries are almost entirely determined by
> `total_study_minutes_per_week`. The preprocessing step above applies **StandardScaler**
> by default - this is why.

### What the Feature Scale control does

| Control | What it does | When to adjust it |
|---|---|---|
| **Show Feature Scale Chart** | Displays selected numeric feature ranges before and after preprocessing side-by-side. | Use it after **Apply Preprocessing** to confirm that normalization has put features on more comparable scales before clustering. |



In [ ]:
# OUTPUT
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

_scale_out = widgets.Output()
_scale_btn = widgets.Button(
    description='Show Feature Scale Chart',
    button_style='info',
    icon='bar-chart',
)

def _feature_range_colour(v):
    if v < 10:
        return '#2ca02c'
    if v < 100:
        return '#ff7f0e'
    if v < 500:
        return '#d62728'
    return '#8b0000'

def _plot_before_after_scales(raw_df, scaled_df):
    raw_numeric = raw_df.select_dtypes(include='number')
    raw_numeric = raw_numeric[[c for c in raw_numeric.columns if 'id' not in c.lower()]]
    shared_cols = [c for c in raw_numeric.columns if c in scaled_df.columns]
    if not shared_cols:
        raise ValueError('No numeric columns are shared between the raw and preprocessed data.')

    raw_ranges = (raw_numeric[shared_cols].max() - raw_numeric[shared_cols].min()).sort_values(ascending=True)
    scaled_ranges = (scaled_df[raw_ranges.index].max() - scaled_df[raw_ranges.index].min())

    fig, axes = plt.subplots(1, 2, figsize=(14, max(5, len(shared_cols) * 0.38)), sharey=True)
    axes[0].barh(raw_ranges.index, raw_ranges.values, color=[_feature_range_colour(v) for v in raw_ranges.values], edgecolor='white')
    axes[1].barh(raw_ranges.index, scaled_ranges.values, color='steelblue', edgecolor='white')

    axes[0].set_title('Before normalization\n(raw feature ranges)', fontsize=11)
    axes[1].set_title('After preprocessing\n(normalized feature ranges)', fontsize=11)
    axes[0].set_xlabel('Raw value range (max - min)')
    axes[1].set_xlabel('Preprocessed value range (max - min)')
    for ax in axes:
        ax.axvline(0, color='black', linewidth=0.5)
        ax.grid(axis='x', alpha=0.2)

    axes[0].legend(
        handles=[
            Patch(facecolor='#2ca02c', label='Range < 10'),
            Patch(facecolor='#ff7f0e', label='Range 10-100'),
            Patch(facecolor='#d62728', label='Range 100-500'),
            Patch(facecolor='#8b0000', label='Range > 500'),
        ],
        loc='lower right',
        fontsize=8,
    )
    fig.suptitle('Feature values before and after normalization', fontsize=13, y=1.02)
    plt.tight_layout()
    return fig, list(raw_ranges.index)

def _show_scales(_btn):
    _scale_out.clear_output(wait=True)
    with _scale_out:
        try:
            df_raw = loader.X_df if loader.X_df is not None else loader.df
            if df_raw is None:
                display(widgets.HTML('<span style="color:red">Load data first.</span>'))
                return
            if getattr(preprocessor, 'X_scaled', None) is None:
                display(widgets.HTML('<span style="color:red">Apply preprocessing first, then show the scale chart.</span>'))
                return
            fig, _ = _plot_before_after_scales(df_raw, preprocessor.X_scaled)
            display(fig)
            plt.close(fig)
        except Exception as exc:
            display(widgets.HTML(f'<span style="color:red">Error: {exc}</span>'))

_scale_btn.on_click(_show_scales)
display(widgets.VBox([_scale_btn, _scale_out]))

## Step 3 — HCA Hyperparameters & Dendrogram

Adjust the controls below to explore different clustering solutions. The dendrogram updates when you click **Compute & Plot**.

| Control | What it does | When to adjust it |
|---|---|---|
| **Sample 500 rows for dendrogram** | Uses a random 500-row subset for the dendrogram while keeping computation readable. | Keep enabled for large datasets; disable only when the dataset is small enough to plot fully. |
| **Colour threshold** | Sets the dendrogram height where branch colors change. It affects visualization, not the stored cluster labels. | Move it to see where natural branch splits appear. |
| **N clusters** | Chooses how many final clusters to assign with `fcluster`. | Set it to the number of groups you want to profile after inspecting the dendrogram. |
| **Truncate at p** | Limits how many leaf/group labels appear at the bottom of the dendrogram. | Lower it for a simpler plot; raise it to see more detail. |
| **Compute & Plot** | Builds the linkage matrix, draws the dendrogram, and assigns cluster labels. | Click after changing any HCA setting. |

This workshop uses **Ward linkage with Euclidean distance** by default. That is a strong general-purpose choice for scaled numeric features because it tends to form compact, interpretable clusters without adding extra configuration decisions.



In [5]:
# HYPERPARAMETERS
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
import matplotlib.pyplot as plt
%matplotlib inline

_hca_out = widgets.Output()
_cluster_out = widgets.Output()
_HCA_LINKAGE = 'ward'
_HCA_METRIC = 'euclidean'

# Sampling controls for large datasets
_sample_check = widgets.Checkbox(
    value=False,
    description='Sample 500 rows for dendrogram (recommended if n > 1000)',
    layout=widgets.Layout(width='480px'),
)

# Threshold & cluster count
_threshold_slider = widgets.FloatSlider(
    value=10.0, min=0.0, max=50.0, step=0.5,
    description='Colour threshold:',
    style={'description_width': '140px'},
    layout=widgets.Layout(width='450px'),
    readout_format='.1f',
)
_n_clusters_slider = widgets.IntSlider(
    value=4, min=2, max=10, step=1,
    description='N clusters:',
    style={'description_width': '140px'},
    layout=widgets.Layout(width='450px'),
)
_truncate_slider = widgets.IntSlider(
    value=30, min=5, max=50, step=1,
    description='Truncate at p:',
    style={'description_width': '140px'},
    layout=widgets.Layout(width='450px'),
)

_run_btn = widgets.Button(
    description='Compute & Plot',
    button_style='primary',
    icon='bar-chart',
)

_Z = None  # linkage matrix cache
_X_used = None

def _run_hca(_btn):
    global _Z, _X_used
    _hca_out.clear_output()
    _cluster_out.clear_output()

    with _hca_out:
        try:
            X_scaled = preprocessor.X_scaled
            if X_scaled is None:
                display(widgets.HTML('<span style="color:red">Run preprocessing first.</span>'))
                return

            n_rows = len(X_scaled)
            if n_rows > 1000:
                display(widgets.HTML(
                    f'<span style="color:orange">⚠ Dataset has {n_rows} rows. '
                    'Dendrogram computation may be slow. Consider enabling sampling above.</span>'
                ))

            X_use = X_scaled
            if _sample_check.value and n_rows > 500:
                X_use = X_scaled.sample(500, random_state=42)
                display(widgets.HTML('<span style="color:blue">ℹ Using 500-row sample for dendrogram.</span>'))

            _X_used = X_use
            display(widgets.HTML('Computing linkage matrix with Ward linkage and Euclidean distance...'))
            _Z = linkage(X_use.values, method=_HCA_LINKAGE, metric=_HCA_METRIC)

            # Adjust threshold slider max to actual data range
            max_dist = float(_Z[:, 2].max())
            _threshold_slider.max = round(max_dist, 1)
            if _threshold_slider.value > max_dist:
                _threshold_slider.value = round(max_dist * 0.3, 1)

            fig = plot_dendrogram(
                _Z,
                truncate_p=_truncate_slider.value,
                color_threshold=_threshold_slider.value,
                title='Dendrogram — Ward linkage, Euclidean distance',
            )
            plt.show()

            # Cluster assignment
            n_clust = _n_clusters_slider.value
            labels = fcluster(_Z, n_clust, criterion='maxclust')
            counts = pd.Series(labels).value_counts().sort_index()
            display(widgets.HTML(f'<b>Cluster sizes ({n_clust} clusters):</b>'))
            display(counts.rename('count').to_frame())

        except Exception as exc:
            display(widgets.HTML(f'<span style="color:red">Error: {exc}</span>'))

def _update_dendrogram(change):
    if _Z is None:
        return
    _hca_out.clear_output()
    with _hca_out:
        try:
            fig = plot_dendrogram(
                _Z,
                truncate_p=_truncate_slider.value,
                color_threshold=_threshold_slider.value,
                title='Dendrogram — Ward linkage, Euclidean distance',
            )
            plt.show()
            n_clust = _n_clusters_slider.value
            labels = fcluster(_Z, n_clust, criterion='maxclust')
            counts = pd.Series(labels).value_counts().sort_index()
            display(widgets.HTML(f'<b>Cluster sizes ({n_clust} clusters):</b>'))
            display(counts.rename('count').to_frame())
        except Exception as exc:
            display(widgets.HTML(f'<span style="color:red">Error: {exc}</span>'))

_threshold_slider.observe(_update_dendrogram, names='value')
_n_clusters_slider.observe(_update_dendrogram, names='value')
_truncate_slider.observe(_update_dendrogram, names='value')
_run_btn.on_click(_run_hca)

display(widgets.VBox([
    widgets.HTML('<h3>🌳 Step 3 — HCA Controls</h3>'),
    widgets.HTML('<span style="color:#666">Using Ward linkage with Euclidean distance.</span>'),
    _sample_check,
    _threshold_slider,
    _n_clusters_slider,
    _truncate_slider,
    _run_btn,
]))
display(_hca_out)

Output()

## Step 4 — Cluster Profiles

After choosing your number of clusters, run this cell to see what each cluster looks like in terms of:
- **Cluster sizes** — how many students are in each group
- **Feature means per cluster** — what distinguishes each group
- **2D scatter plot** — students projected into 2 dimensions (PCA) and coloured by cluster

### What the Cluster Profile control does

| Control | What it does | When to adjust it |
|---|---|---|
| **Show Cluster Profiles** | Summarises the clusters created in Step 3 using counts, average feature values, and a PCA scatter plot. | Click after recomputing clusters or changing **N clusters**. |



In [6]:
# OUTPUT
_profile_out = widgets.Output()
_profile_btn = widgets.Button(
    description='Show Cluster Profiles',
    button_style='info',
    icon='table',
)

def _show_profiles(_btn):
    _profile_out.clear_output()
    with _profile_out:
        try:
            if _Z is None or _X_used is None:
                display(widgets.HTML('<span style="color:red">Run Step 3 first.</span>'))
                return

            n_clust = _n_clusters_slider.value
            labels = fcluster(_Z, n_clust, criterion='maxclust')

            # Cluster size table
            counts = pd.Series(labels, name='cluster').value_counts().sort_index()
            display(widgets.HTML('<b>Cluster sizes:</b>'))
            display(counts.rename('n_students').to_frame())

            # Feature means heatmap (on original scale if possible)
            X_orig = loader.X_df.copy() if loader.X_df is not None else _X_used
            X_orig = X_orig.select_dtypes(include='number').loc[_X_used.index]
            cluster_means = X_orig.groupby(labels).mean()
            cluster_means.index = [f'Cluster {i}' for i in cluster_means.index]

            display(widgets.HTML('<b>Mean feature values per cluster:</b>'))
            fig = plot_cluster_profile_heatmap(cluster_means)
            plt.show()

            # 2D scatter
            fig2 = plot_cluster_scatter(
                _X_used,
                labels,
                method='PCA',
                title=f'Student Clusters — PCA projection ({n_clust} clusters)',
            )
            plt.show()

        except Exception as exc:
            display(widgets.HTML(f'<span style="color:red">Error: {exc}</span>'))

_profile_btn.on_click(_show_profiles)
display(widgets.VBox([_profile_btn, _profile_out]))

## After the Workshop: Reusing This HCA Notebook

Use this notebook when you want to explore whether your data contains natural groups, especially when you do not already know what the categories should be.

HCA is useful for:

- discovering possible participant or student archetypes
- comparing different numbers of clusters
- explaining cluster structure visually with a dendrogram
- deciding whether K-Means is worth trying next

### What to Save

For your own analysis notes, save:

- the feature columns used
- the scaling method
- the fixed HCA defaults used: Ward linkage and Euclidean distance
- the chosen number of clusters
- the dendrogram
- the cluster profile table or heatmap
- your written interpretation of each cluster